# 1. Install Required Packages

This section will install all essential packages for both backend (Flask, MongoDB, CORS, dotenv) and frontend (React, Vite, axios, react-router-dom).


In [21]:
# Install backend packages
!pip install flask pymongo flask-cors python-dotenv

## Frontend (React + Vite) Packages

To set up the frontend, run the following commands in your terminal:

```sh
npm create vite@latest frontend -- --template react
cd frontend
npm install axios react-router-dom
```


# 2. Initialize New Workspace Directory

Create a new directory for your project workspace and navigate into it.

# 3. Set Up Version Control (Git)

Initialize a Git repository and create a `.gitignore` file to exclude unnecessary files from version control.


In [22]:
# Initialize git repository (run in terminal)
!git init

# Create .gitignore for Python, Node, and general files
with open('.gitignore', 'w') as f:
    f.write('''# Python
__pycache__/
*.pyc
.env

# Node
node_modules/
dist/
build/

# General
.DS_Store
.vscode/
''')


Reinitialized existing Git repository in C:/Users/91799/OneDrive/Desktop/EDU_Moon/student-colab/.git/


# 4. Set Up Python Virtual Environment

Create and activate a virtual environment for the backend to manage dependencies.


In [23]:
# Create a virtual environment (Windows)
!python -m venv venv

# Activate the virtual environment (run in terminal, not in notebook)
# .\venv\Scripts\activate

# Install backend dependencies inside the virtual environment
!pip install flask pymongo flask-cors python-dotenv


# 5. Scaffold Project Structure

Create the backend and frontend folders if they do not exist.


In [24]:
import os

# Create backend and frontend directories
os.makedirs('backend', exist_ok=True)
os.makedirs('frontend', exist_ok=True)

# Create placeholder README files
with open('backend/README.md', 'w') as f:
    f.write('# Backend\nFlask REST API for Student Collaboration Project.\n')
with open('frontend/README.md', 'w') as f:
    f.write('# Frontend\nReact app for Student Collaboration Project.\n')


# 6. Scaffold Backend (Flask API)

Create the initial backend files: `app.py`, `requirements.txt`, and `.env.example`.


In [25]:
# Create backend/app.py
with open('backend/app.py', 'w') as f:
    f.write('''from flask import Flask\nfrom flask_cors import CORS\nfrom dotenv import load_dotenv\nimport os\n\nload_dotenv()\n\napp = Flask(__name__)\nCORS(app)\n\n@app.route("/")\ndef home():\n    return {"message": "Student Collaboration API is running."}\n\nif __name__ == "__main__":\n    app.run(debug=True)\n''')

# Create backend/requirements.txt
with open('backend/requirements.txt', 'w') as f:
    f.write('flask\npymongo\nflask-cors\npython-dotenv\n')

# Create backend/.env.example
with open('backend/.env.example', 'w') as f:
    f.write('''MONGO_URI=mongodb://localhost:27017/student_collab\nSECRET_KEY=your_secret_key\n''')


# 7. Scaffold Frontend (React + Vite)

Follow these commands in your terminal to set up the frontend:

```sh
npm create vite@latest frontend -- --template react
cd frontend
npm install axios react-router-dom
```

This will create the React app in the `frontend` folder and install the required packages.


# 8. Plan Backend API Structure

Define the main API endpoints for user authentication, posts, registration, and profile management. This will guide the implementation of Flask routes and MongoDB collections.

**Main Endpoints:**
- `POST /api/auth/signup` — Register a new user
- `POST /api/auth/login` — User login
- `GET /api/posts` — List all posts
- `POST /api/posts` — Create a new post
- `POST /api/posts/<post_id>/register` — Register for a post
- `GET /api/profile` — Get current user profile
- `PUT /api/profile` — Update user profile

Next, we will scaffold the backend folder structure and create the initial Flask blueprints and MongoDB connection.


In [26]:
import os

# Create backend subfolders
os.makedirs('backend/routes', exist_ok=True)
os.makedirs('backend/models', exist_ok=True)
os.makedirs('backend/utils', exist_ok=True)

# Create __init__.py files
open('backend/routes/__init__.py', 'a').close()
open('backend/models/__init__.py', 'a').close()
open('backend/utils/__init__.py', 'a').close()

# Create blueprint files
for name in ['auth.py', 'posts.py', 'profile.py']:
    with open(f'backend/routes/{name}', 'w') as f:
        f.write(f"""from flask import Blueprint\n\n{name.split('.')[0]}_bp = Blueprint('{name.split('.')[0]}', __name__)\n\n# Add your routes here\n""")

# Create MongoDB connection utility
with open('backend/utils/db.py', 'w') as f:
    f.write('''from flask import current_app, g\nfrom pymongo import MongoClient\nimport os\n\ndef get_db():\n    if 'db' not in g:\n        mongo_uri = os.getenv('MONGO_URI')\n        client = MongoClient(mongo_uri)\n        g.db = client.get_default_database()\n    return g.db\n''')


# 9. Wire Up Flask App and Blueprints

Update `backend/app.py` to register blueprints for authentication, posts, and profile, and connect to MongoDB using the utility function.


In [27]:
# Update backend/app.py to register blueprints and use MongoDB connection
with open('backend/app.py', 'w') as f:
    f.write('''from flask import Flask
from flask_cors import CORS
from dotenv import load_dotenv
import os

from routes.auth import auth_bp
from routes.posts import posts_bp
from routes.profile import profile_bp
from utils.db import get_db

load_dotenv()

app = Flask(__name__)
CORS(app)

# Register blueprints
app.register_blueprint(auth_bp, url_prefix='/api/auth')
app.register_blueprint(posts_bp, url_prefix='/api/posts')
app.register_blueprint(profile_bp, url_prefix='/api/profile')

@app.route("/")
def home():
    return {"message": "Student Collaboration API is running."}

if __name__ == "__main__":
    app.run(debug=True)
''')


# 10. Implement Authentication Endpoints

Add user signup and login logic to `backend/routes/auth.py`. This will use MongoDB for storing user data and Python's `werkzeug.security` for password hashing.


In [28]:
# Implement signup and login endpoints in backend/routes/auth.py
with open('backend/routes/auth.py', 'w') as f:
    f.write('''from flask import Blueprint, request, jsonify, session
from werkzeug.security import generate_password_hash, check_password_hash
from utils.db import get_db
import os

auth_bp = Blueprint('auth', __name__)

@auth_bp.route('/signup', methods=['POST'])
def signup():
    data = request.json
    db = get_db()
    if db.users.find_one({'email': data['email']}):
        return jsonify({'error': 'Email already registered'}), 400
    hashed_pw = generate_password_hash(data['password'])
    user = {
        'email': data['email'],
        'password': hashed_pw,
        'name': data.get('name', ''),
        'role': data.get('role', 'student')
    }
    db.users.insert_one(user)
    return jsonify({'message': 'User registered successfully'})

@auth_bp.route('/login', methods=['POST'])
def login():
    data = request.json
    db = get_db()
    user = db.users.find_one({'email': data['email']})
    if not user or not check_password_hash(user['password'], data['password']):
        return jsonify({'error': 'Invalid credentials'}), 401
    # For demo: return user info (no JWT/session for now)
    return jsonify({'message': 'Login successful', 'user': {'email': user['email'], 'name': user.get('name', ''), 'role': user.get('role', '')}})
''')


# 11. Implement Post Endpoints

Add endpoints to create a post, list all posts, and register for a post in `backend/routes/posts.py`.


In [29]:
# Implement post endpoints in backend/routes/posts.py
with open('backend/routes/posts.py', 'w') as f:
    f.write('''from flask import Blueprint, request, jsonify
from utils.db import get_db
from bson import ObjectId

posts_bp = Blueprint('posts', __name__)

@posts_bp.route('/', methods=['GET'])
def get_posts():
    db = get_db()
    posts = list(db.posts.find())
    for post in posts:
        post['_id'] = str(post['_id'])
    return jsonify(posts)

@posts_bp.route('/', methods=['POST'])
def create_post():
    data = request.json
    db = get_db()
    post = {
        'title': data['title'],
        'description': data.get('description', ''),
        'created_by': data.get('created_by', ''),
        'registrations': []
    }
    result = db.posts.insert_one(post)
    post['_id'] = str(result.inserted_id)
    return jsonify(post), 201

@posts_bp.route('/<post_id>/register', methods=['POST'])
def register_for_post(post_id):
    data = request.json
    db = get_db()
    user_email = data.get('email')
    post = db.posts.find_one({'_id': ObjectId(post_id)})
    if not post:
        return jsonify({'error': 'Post not found'}), 404
    if user_email in post.get('registrations', []):
        return jsonify({'error': 'Already registered'}), 400
    db.posts.update_one({'_id': ObjectId(post_id)}, {'$push': {'registrations': user_email}})
    return jsonify({'message': 'Registered successfully'})
''')


# 12. Implement User Profile Endpoints

Add endpoints to get and update the current user's profile in `backend/routes/profile.py`.


In [30]:
# Implement user profile endpoints in backend/routes/profile.py
with open('backend/routes/profile.py', 'w') as f:
    f.write('''from flask import Blueprint, request, jsonify
from utils.db import get_db

profile_bp = Blueprint('profile', __name__)

@profile_bp.route('/', methods=['GET'])
def get_profile():
    email = request.args.get('email')
    db = get_db()
    user = db.users.find_one({'email': email}, {'password': 0})
    if not user:
        return jsonify({'error': 'User not found'}), 404
    user['_id'] = str(user['_id'])
    return jsonify(user)

@profile_bp.route('/', methods=['PUT'])
def update_profile():
    data = request.json
    email = data.get('email')
    db = get_db()
    update_fields = {k: v for k, v in data.items() if k != 'email' and k != 'password'}
    result = db.users.update_one({'email': email}, {'$set': update_fields})
    if result.matched_count == 0:
        return jsonify({'error': 'User not found'}), 404
    return jsonify({'message': 'Profile updated'})
''')


# 13. Plan Frontend React App Structure

Define the main pages and components for the React frontend:

**Pages:**
- Login
- Signup
- Dashboard (list posts)
- Create Post
- Register for Post
- Profile

**Common Components:**
- Navbar
- ProtectedRoute (for authenticated pages)

Next, we will scaffold the folder structure and starter files for these pages and components.


In [31]:
import os

# Create frontend/src/pages and frontend/src/components directories
os.makedirs('frontend/src/pages', exist_ok=True)
os.makedirs('frontend/src/components', exist_ok=True)

# Create starter files for pages
for page in ['Login.jsx', 'Signup.jsx', 'Dashboard.jsx', 'CreatePost.jsx', 'RegisterPost.jsx', 'Profile.jsx']:
    with open(f'frontend/src/pages/{page}', 'w') as f:
        f.write(f"""import React from 'react';\n\nexport default function {page.split('.')[0]}() {{\n    return <div>{page.split('.')[0]} Page</div>;\n}}\n""")

# Create starter files for components
for comp in ['Navbar.jsx', 'ProtectedRoute.jsx']:
    with open(f'frontend/src/components/{comp}', 'w') as f:
        f.write(f"""import React from 'react';\n\nexport default function {comp.split('.')[0]}() {{\n    return <div>{comp.split('.')[0]}</div>;\n}}\n""")


# 14. Set Up React Router and Main App Structure

Configure routing in `frontend/src/App.jsx` to connect all pages, and use the Navbar and ProtectedRoute components.


In [32]:
# Create frontend/src/App.jsx with routing and navbar
with open('frontend/src/App.jsx', 'w') as f:
    f.write('''import React from 'react';
import { BrowserRouter as Router, Routes, Route } from 'react-router-dom';
import Navbar from './components/Navbar';
import ProtectedRoute from './components/ProtectedRoute';
import Login from './pages/Login';
import Signup from './pages/Signup';
import Dashboard from './pages/Dashboard';
import CreatePost from './pages/CreatePost';
import RegisterPost from './pages/RegisterPost';
import Profile from './pages/Profile';

export default function App() {
  return (
    <Router>
      <Navbar />
      <Routes>
        <Route path="/login" element={<Login />} />
        <Route path="/signup" element={<Signup />} />
        <Route path="/" element={<ProtectedRoute><Dashboard /></ProtectedRoute>} />
        <Route path="/create" element={<ProtectedRoute><CreatePost /></ProtectedRoute>} />
        <Route path="/register/:id" element={<ProtectedRoute><RegisterPost /></ProtectedRoute>} />
        <Route path="/profile" element={<ProtectedRoute><Profile /></ProtectedRoute>} />
      </Routes>
    </Router>
  );
}
''')


# 15. Implement Authentication Logic (Login & Signup)

Add form handling, axios API calls, and local storage for authentication in `Login.jsx` and `Signup.jsx`.


In [33]:
# Implement Login.jsx with form, axios call, and localStorage
with open('frontend/src/pages/Login.jsx', 'w') as f:
    f.write('''import React, { useState } from 'react';
import axios from 'axios';
import { useNavigate } from 'react-router-dom';

export default function Login() {
  const [email, setEmail] = useState('');
  const [password, setPassword] = useState('');
  const [error, setError] = useState('');
  const navigate = useNavigate();

  const handleSubmit = async (e) => {
    e.preventDefault();
    setError('');
    try {
      const res = await axios.post('http://localhost:5000/api/auth/login', { email, password });
      localStorage.setItem('user', JSON.stringify(res.data.user));
      navigate('/');
    } catch (err) {
      setError(err.response?.data?.error || 'Login failed');
    }
  };

  return (
    <div>
      <h2>Login</h2>
      <form onSubmit={handleSubmit}>
        <input type="email" placeholder="Email" value={email} onChange={e => setEmail(e.target.value)} required />
        <input type="password" placeholder="Password" value={password} onChange={e => setPassword(e.target.value)} required />
        <button type="submit">Login</button>
      </form>
      {error && <div style={{color:'red'}}>{error}</div>}
    </div>
  );
}
''')


In [34]:
# Implement Signup.jsx with form, axios call, and localStorage
with open('frontend/src/pages/Signup.jsx', 'w') as f:
    f.write('''import React, { useState } from 'react';
import axios from 'axios';
import { useNavigate } from 'react-router-dom';

export default function Signup() {
  const [email, setEmail] = useState('');
  const [password, setPassword] = useState('');
  const [name, setName] = useState('');
  const [error, setError] = useState('');
  const navigate = useNavigate();

  const handleSubmit = async (e) => {
    e.preventDefault();
    setError('');
    try {
      await axios.post('http://localhost:5000/api/auth/signup', { email, password, name });
      navigate('/login');
    } catch (err) {
      setError(err.response?.data?.error || 'Signup failed');
    }
  };

  return (
    <div>
      <h2>Signup</h2>
      <form onSubmit={handleSubmit}>
        <input type="text" placeholder="Name" value={name} onChange={e => setName(e.target.value)} required />
        <input type="email" placeholder="Email" value={email} onChange={e => setEmail(e.target.value)} required />
        <input type="password" placeholder="Password" value={password} onChange={e => setPassword(e.target.value)} required />
        <button type="submit">Signup</button>
      </form>
      {error && <div style={{color:'red'}}>{error}</div>}
    </div>
  );
}
''')


In [35]:
# Implement ProtectedRoute.jsx for route protection
with open('frontend/src/components/ProtectedRoute.jsx', 'w') as f:
    f.write('''import React from 'react';
import { Navigate } from 'react-router-dom';

export default function ProtectedRoute({ children }) {
  const user = localStorage.getItem('user');
  if (!user) return <Navigate to="/login" />;
  return children;
}
''')


In [36]:
# Implement Navbar.jsx with navigation links and logout
with open('frontend/src/components/Navbar.jsx', 'w') as f:
    f.write('''import React from 'react';
import { Link, useNavigate } from 'react-router-dom';

export default function Navbar() {
  const user = JSON.parse(localStorage.getItem('user'));
  const navigate = useNavigate();
  const handleLogout = () => {
    localStorage.removeItem('user');
    navigate('/login');
  };
  return (
    <nav>
      <Link to="/">Dashboard</Link> |
      <Link to="/create">Create Post</Link> |
      <Link to="/profile">Profile</Link> |
      {user ? (
        <span onClick={handleLogout} style={{cursor:'pointer'}}>Logout</span>
      ) : (
        <>
          <Link to="/login">Login</Link> |
          <Link to="/signup">Signup</Link>
        </>
      )}
    </nav>
  );
}
''')


In [37]:
# Implement Dashboard.jsx to fetch and display posts
with open('frontend/src/pages/Dashboard.jsx', 'w') as f:
    f.write('''import React, { useEffect, useState } from 'react';
import axios from 'axios';
import { Link } from 'react-router-dom';

export default function Dashboard() {
  const [posts, setPosts] = useState([]);
  useEffect(() => {
    axios.get('http://localhost:5000/api/posts/')
      .then(res => setPosts(res.data))
      .catch(() => setPosts([]));
  }, []);
  return (
    <div>
      <h2>Posts</h2>
      <ul>
        {posts.map(post => (
          <li key={post._id}>
            <b>{post.title}</b> - {post.description} <br />
            <Link to={`/register/${post._id}`}>Register</Link>
          </li>
        ))}
      </ul>
    </div>
  );
}
''')


In [38]:
# Implement CreatePost.jsx with form to create a new post
with open('frontend/src/pages/CreatePost.jsx', 'w') as f:
    f.write('''import React, { useState } from 'react';
import axios from 'axios';
import { useNavigate } from 'react-router-dom';

export default function CreatePost() {
  const [title, setTitle] = useState('');
  const [description, setDescription] = useState('');
  const [error, setError] = useState('');
  const navigate = useNavigate();
  const user = JSON.parse(localStorage.getItem('user'));

  const handleSubmit = async (e) => {
    e.preventDefault();
    setError('');
    try {
      await axios.post('http://localhost:5000/api/posts/', {
        title,
        description,
        created_by: user?.email || ''
      });
      navigate('/');
    } catch (err) {
      setError('Failed to create post');
    }
  };

  return (
    <div>
      <h2>Create Post</h2>
      <form onSubmit={handleSubmit}>
        <input type="text" placeholder="Title" value={title} onChange={e => setTitle(e.target.value)} required />
        <input type="text" placeholder="Description" value={description} onChange={e => setDescription(e.target.value)} />
        <button type="submit">Create</button>
      </form>
      {error && <div style={{color:'red'}}>{error}</div>}
    </div>
  );
}
''')


In [39]:
# Implement RegisterPost.jsx to register for a post
with open('frontend/src/pages/RegisterPost.jsx', 'w') as f:
    f.write('''import React, { useState } from 'react';
import axios from 'axios';
import { useParams, useNavigate } from 'react-router-dom';

export default function RegisterPost() {
  const { id } = useParams();
  const [message, setMessage] = useState('');
  const user = JSON.parse(localStorage.getItem('user'));
  const navigate = useNavigate();

  const handleRegister = async () => {
    try {
      await axios.post(`http://localhost:5000/api/posts/${id}/register`, { email: user?.email });
      setMessage('Registered successfully!');
      setTimeout(() => navigate('/'), 1000);
    } catch (err) {
      setMessage(err.response?.data?.error || 'Registration failed');
    }
  };

  return (
    <div>
      <h2>Register for Post</h2>
      <button onClick={handleRegister}>Register</button>
      {message && <div>{message}</div>}
    </div>
  );
}
''')


In [40]:
# Implement Profile.jsx to fetch and update user profile
with open('frontend/src/pages/Profile.jsx', 'w') as f:
    f.write('''import React, { useEffect, useState } from 'react';
import axios from 'axios';

export default function Profile() {
  const [profile, setProfile] = useState({});
  const [name, setName] = useState('');
  const [message, setMessage] = useState('');
  const user = JSON.parse(localStorage.getItem('user'));

  useEffect(() => {
    axios.get('http://localhost:5000/api/profile/', { params: { email: user?.email } })
      .then(res => {
        setProfile(res.data);
        setName(res.data.name || '');
      });
  }, [user]);

  const handleUpdate = async (e) => {
    e.preventDefault();
    try {
      await axios.put('http://localhost:5000/api/profile/', { email: user?.email, name });
      setMessage('Profile updated');
    } catch {
      setMessage('Update failed');
    }
  };

  return (
    <div>
      <h2>Profile</h2>
      <form onSubmit={handleUpdate}>
        <input type="text" value={name} onChange={e => setName(e.target.value)} />
        <button type="submit">Update</button>
      </form>
      {message && <div>{message}</div>}
      <div>Email: {profile.email}</div>
      <div>Role: {profile.role}</div>
    </div>
  );
}
''')


# 16. Run and Test Your Full-Stack App

- Start MongoDB locally (or use a cloud MongoDB URI in your `.env` file).
- In one terminal, run the backend:
  ```sh
  cd backend
  # activate venv if needed
  flask run
  ```
- In another terminal, run the frontend:
  ```sh
  cd frontend
  npm run dev
  ```
- Open the app in your browser at `http://localhost:5173` (or the port shown).

Test all features: authentication, post creation, registration, and profile update.
